# Energy & Convergence Analysis for RMT Simulations

Implements the error norms from the paper (eqs 41–45):
- `E_ke = |ke(N) − ke_ref|`
- `E_se = |se(N) − se_ref|`
- `E_v  = ‖|v|_N − |v|_ref‖_∞`
- `E_p  = ‖p_N − p_ref‖_∞`
- `E_ξ  = ‖|ξ|_N − |ξ|_ref‖_∞`

Reference solution: 1024×1024 grid (`_2` run series).  
Coarser fields are compared by striding the 1024 reference (grids align exactly since 1024/N is always an integer).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import h5py
import os

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

OUTPUT_BASE = os.path.abspath('../outputs')

# '_2' run series: 64, 128, 256, 512, 1024 — all complete at t≈1
RUN_DIRS = {
    64:   'output_taylor_green_soft_disc_64x64_2',
    128:  'output_taylor_green_soft_disc_128x128_2',
    256:  'output_taylor_green_soft_disc_256x256_2',
    512:  'output_taylor_green_soft_disc_512x512_2',
    1024: 'output_taylor_green_soft_disc_1024x1024_2',
}

REF_N   = 1024
TEST_Ns = [64, 128, 256, 512]

def last_h5(dirpath):
    """Return path to the last (highest-step) HDF5 file in a directory."""
    files = sorted(f for f in os.listdir(dirpath) if f.endswith('.h5'))
    return os.path.join(dirpath, files[-1])

def load_h5(N):
    path = os.path.join(OUTPUT_BASE, RUN_DIRS[N])
    fpath = last_h5(path)
    with h5py.File(fpath, 'r') as f:
        data = {
            'ke': float(f.attrs['kinetic_energy']),
            'se': float(f.attrs['strain_energy']),
            'time': float(f.attrs['time']),
            'a':  f['a'][:],
            'b':  f['b'][:],
            'p':  f['p'][:],
            'X1': f['X1'][:],
            'X2': f['X2'][:],
        }
    return data

print('Checking available data...')
for N, d in RUN_DIRS.items():
    full = os.path.join(OUTPUT_BASE, d)
    if os.path.exists(full):
        files = sorted(f for f in os.listdir(full) if f.endswith('.h5'))
        print(f'  N={N:4d}: {len(files):3d} HDF5 files')
    else:
        print(f'  N={N:4d}: NOT FOUND')

In [ ]:
# ── KE and SE vs time ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = {64: 'C0', 128: 'C1', 256: 'C2', 512: 'C3', 1024: 'C4'}

for N, dirname in RUN_DIRS.items():
    csv_path = os.path.join(OUTPUT_BASE, dirname, 'energy_history.csv')
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    lw = 2.0 if N == REF_N else 1.0
    ax1.plot(df['time'], df['kinetic_energy'], color=colors[N], lw=lw, label=f'{N}×{N}')
    ax2.plot(df['time'], df['strain_energy'],  color=colors[N], lw=lw, label=f'{N}×{N}')

for ax, title, ylabel in [
    (ax1, 'Kinetic Energy vs Time',  'KE'),
    (ax2, 'Strain Energy vs Time',   'SE'),
]:
    ax.set_xlabel('t')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False)
    ax.grid(True, ls='--', alpha=0.4)

fig.tight_layout()
plt.savefig('energy_vs_time.png', bbox_inches='tight')
plt.show()

In [ ]:

# ── Part A: time-averaged energy errors from CSVs (E_ke, E_se) ───────────────
print('Loading energy time series from CSVs...')

csv_data = {}
for N, dirname in RUN_DIRS.items():
    csv_path = os.path.join(OUTPUT_BASE, dirname, 'energy_history.csv')
    if os.path.exists(csv_path):
        csv_data[N] = pd.read_csv(csv_path)
        print(f'  N={N:4d}: {len(csv_data[N])} rows, t∈[{csv_data[N]["time"].min():.3f}, {csv_data[N]["time"].max():.3f}]')

ref_df    = csv_data[REF_N]
t_common  = ref_df['time'].values
Ns        = np.array(TEST_Ns)

E_ke_avg  = np.zeros(len(Ns))
E_se_avg  = np.zeros(len(Ns))
E_ke_t1   = np.zeros(len(Ns))
E_se_t1   = np.zeros(len(Ns))

ref_ke_interp = np.interp(t_common, ref_df['time'], ref_df['kinetic_energy'])
ref_se_interp = np.interp(t_common, ref_df['time'], ref_df['strain_energy'])

print(f'\n{"N":>6}  {"E_ke_avg":>12}  {"E_se_avg":>12}  {"E_ke(t=1)":>12}  {"E_se(t=1)":>12}')
for idx, N in enumerate(TEST_Ns):
    df = csv_data[N]
    ke_interp = np.interp(t_common, df['time'], df['kinetic_energy'])
    se_interp = np.interp(t_common, df['time'], df['strain_energy'])
    T = t_common[-1] - t_common[0]
    E_ke_avg[idx] = np.trapz(np.abs(ke_interp - ref_ke_interp), t_common) / T
    E_se_avg[idx] = np.trapz(np.abs(se_interp - ref_se_interp), t_common) / T
    E_ke_t1[idx]  = abs(df['kinetic_energy'].iloc[-1] - ref_df['kinetic_energy'].iloc[-1])
    E_se_t1[idx]  = abs(df['strain_energy'].iloc[-1]  - ref_df['strain_energy'].iloc[-1])
    print(f'{N:6d}  {E_ke_avg[idx]:12.4e}  {E_se_avg[idx]:12.4e}  '
          f'{E_ke_t1[idx]:12.4e}  {E_se_t1[idx]:12.4e}')

# ── Part B: point-in-time field errors from HDF5 (E_v, E_p, E_xi) ────────────
# Each HDF5 snapshot is at the final time of that run.  Because dt ∝ 1/N the
# runs end at slightly different times; we accept this small discrepancy and
# note the final time for each resolution.

print('\nLoading final HDF5 snapshots for field convergence...')
h5_data = {}
for N in list(TEST_Ns) + [REF_N]:
    try:
        h5_data[N] = load_h5(N)
        print(f'  N={N:4d}: t={h5_data[N]["time"]:.4f}')
    except Exception as e:
        print(f'  N={N:4d}: FAILED ({e})')

ref = h5_data[REF_N]
ref_vmag = np.sqrt(ref['a']**2  + ref['b']**2)
ref_xmag = np.sqrt(ref['X1']**2 + ref['X2']**2)

E_v   = np.zeros(len(Ns))
E_p   = np.zeros(len(Ns))
E_xi  = np.zeros(len(Ns))

print(f'\n{"N":>6}  {"E_v (∞-norm)":>14}  {"E_p (∞-norm)":>14}  {"E_xi (∞-norm)":>14}')
for idx, N in enumerate(TEST_Ns):
    if N not in h5_data:
        continue
    s = REF_N // N       # stride: 1024/N is always an integer
    d = h5_data[N]

    vmag_N = np.sqrt(d['a']**2  + d['b']**2)
    xmag_N = np.sqrt(d['X1']**2 + d['X2']**2)

    E_v[idx]  = np.max(np.abs(vmag_N - ref_vmag[::s, ::s]))
    E_p[idx]  = np.max(np.abs(d['p'] - ref['p'][::s, ::s]))
    E_xi[idx] = np.max(np.abs(xmag_N  - ref_xmag[::s, ::s]))

    print(f'{N:6d}  {E_v[idx]:14.4e}  {E_p[idx]:14.4e}  {E_xi[idx]:14.4e}')

np.savez('convergence_data.npz',
         Ns=Ns,
         E_ke_avg=E_ke_avg, E_se_avg=E_se_avg,
         E_ke_t1=E_ke_t1,   E_se_t1=E_se_t1,
         E_v=E_v,            E_p=E_p,           E_xi=E_xi)
print('\nSaved convergence_data.npz')


In [ ]:

# ── Convergence plots ─────────────────────────────────────────────────────────
def convergence_orders(Ns, errors):
    orders = []
    for i in range(len(Ns) - 1):
        if errors[i] > 0 and errors[i+1] > 0:
            p = np.log(errors[i] / errors[i+1]) / np.log(Ns[i+1] / Ns[i])
        else:
            p = float('nan')
        orders.append(p)
    return orders

def add_ref_slopes(ax, h_line, y_anchor):
    ax.loglog(h_line, y_anchor * (h_line[0] / h_line)**1, 'k:', lw=1, alpha=0.5, label='O(h¹)')
    ax.loglog(h_line, y_anchor * (h_line[0] / h_line)**2, 'k--', lw=1, alpha=0.5, label='O(h²)')

def annotate_orders(ax, Ns, errors, orders, color, offset=(5, 5)):
    for i, p in enumerate(orders):
        if not np.isnan(p):
            xm = np.sqrt(Ns[i] * Ns[i+1])
            ym = np.sqrt(errors[i] * errors[i+1])
            ax.annotate(f'{p:.2f}', (xm, ym), fontsize=8, color=color,
                        xytext=offset, textcoords='offset points')

h_line = np.array([Ns[0], Ns[-1]], dtype=float)

# ── Row 1: E_ke and E_se (time-averaged vs point-in-time) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Scalar Energy Convergence', fontsize=13)

ax = axes[0]
ax.loglog(Ns, E_ke_t1, 'o-', color='C0', lw=1.5, ms=6, label='E_ke  (t=1, point)')
ax.loglog(Ns, E_se_t1, 's-', color='C1', lw=1.5, ms=6, label='E_se  (t=1, point)')
add_ref_slopes(ax, h_line, E_ke_t1[0])
annotate_orders(ax, Ns, E_ke_t1, convergence_orders(Ns, E_ke_t1), 'C0', offset=(5,  5))
annotate_orders(ax, Ns, E_se_t1, convergence_orders(Ns, E_se_t1), 'C1', offset=(5, -12))
ax.set(xlabel='N', ylabel='Error', title='Point-in-time at t=1\n(phase mismatch distorts order)')
ax.legend(frameon=False); ax.grid(True, which='both', ls='--', alpha=0.3)

ax = axes[1]
ax.loglog(Ns, E_ke_avg, 'o-', color='C0', lw=1.5, ms=6, label='E_ke  (time-avg)')
ax.loglog(Ns, E_se_avg, 's-', color='C1', lw=1.5, ms=6, label='E_se  (time-avg)')
add_ref_slopes(ax, h_line, E_ke_avg[0])
annotate_orders(ax, Ns, E_ke_avg, convergence_orders(Ns, E_ke_avg), 'C0', offset=(5,  5))
annotate_orders(ax, Ns, E_se_avg, convergence_orders(Ns, E_se_avg), 'C1', offset=(5, -12))
ax.set(xlabel='N', ylabel='Error', title='Time-averaged  E = (1/T)∫|e(t,N)−e(t,1024)| dt')
ax.legend(frameon=False); ax.grid(True, which='both', ls='--', alpha=0.3)

fig.tight_layout()
plt.savefig('convergence_energy.png', bbox_inches='tight')
plt.show()

# ── Row 2: E_v, E_p, E_xi (infinity-norm, point-in-time from HDF5) ───────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Primitive Variable Convergence  (∞-norm, final snapshot)', fontsize=13)

specs = [
    (axes[0], E_v,  'C2', 'o', r'$E_v = \|\,|v|_N - |v|_{ref}\,\|_\infty$',  'E_v'),
    (axes[1], E_p,  'C3', 's', r'$E_p = \|p_N - p_{ref}\|_\infty$',            'E_p'),
    (axes[2], E_xi, 'C4', '^', r'$E_\xi = \|\,|\xi|_N - |\xi|_{ref}\,\|_\infty$', 'E_xi'),
]

for ax, err, color, marker, label, name in specs:
    ax.loglog(Ns, err, f'{marker}-', color=color, lw=1.5, ms=6, label=label)
    add_ref_slopes(ax, h_line, err[0])
    orders = convergence_orders(Ns, err)
    annotate_orders(ax, Ns, err, orders, color, offset=(5, 5))
    ax.set(xlabel='N', ylabel='Error', title=label)
    ax.legend(frameon=False, fontsize=8)
    ax.grid(True, which='both', ls='--', alpha=0.3)
    print(f'{name} orders: {[f"{p:.2f}" for p in orders]}')

fig.tight_layout()
plt.savefig('convergence_primitives.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Total energy conservation check ──────────────────────────────────────────
# E = KE + SE + ∫ε dt should be approximately constant in time.
# The paper reports < 1% change from t=0 to t=1.

fig, ax = plt.subplots(figsize=(8, 4))

for N, dirname in RUN_DIRS.items():
    csv_path = os.path.join(OUTPUT_BASE, dirname, 'energy_history.csv')
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    E0 = df['total_energy'].iloc[0]
    rel_change = (df['total_energy'] - E0) / E0 * 100.0  # percent
    lw = 2.0 if N == REF_N else 1.0
    ax.plot(df['time'], rel_change, color=colors[N], lw=lw, label=f'{N}×{N}')

ax.axhline(0, color='k', lw=0.5)
ax.axhline(-1, color='k', ls=':', lw=0.8, alpha=0.5, label='±1% band')
ax.axhline(+1, color='k', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel('t')
ax.set_ylabel('ΔE / E₀  [%]')
ax.set_title('Total Energy Conservation  (E = KE + SE + ∫ε dt)')
ax.legend(frameon=False)
ax.grid(True, ls='--', alpha=0.4)
fig.tight_layout()
plt.savefig('energy_conservation.png', bbox_inches='tight')
plt.show()